## **Proceso Carga Data Warehouse**

### **1. Carga de dimensiones**

In [1]:
import pandas as pd
import numpy as np

from sqlalchemy import create_engine

In [2]:
USUARIO = "postgres"
PASSWORD = "1028"
HOST = "localhost"
PUERTO = "5432"
BASE_DATOS = "dw_analisis_criminalidad"

engine = create_engine(
    f"postgresql+psycopg2://{USUARIO}:{PASSWORD}@{HOST}:{PUERTO}/{BASE_DATOS}"
)

print("Conexión creada correctamente.")

Conexión creada correctamente.


In [3]:
ruta = "Salidas/delitos_transformados.csv"

df = pd.read_csv(
    ruta,
    parse_dates=["fecha_hecho"],
    encoding="utf-8-sig"
)

print(df.head())
print()
print(df.info())

  fecha_hecho departamento  municipio               delito  \
0  2014-01-01    Antioquia      Amagá             Amenazas   
1  2014-01-01    Antioquia      Amagá  Lesiones personales   
2  2014-01-01    Antioquia      Andes  Lesiones personales   
3  2014-01-01    Antioquia  Angostura  Lesiones personales   
4  2014-01-01    Antioquia  Angostura  Lesiones personales   

                  armas_medios     genero grupo_etario  cantidad  
0          SIN EMPLEO DE ARMAS   Femenino      Adultos         1  
1  ARMA BLANCA / CORTOPUNZANTE  Masculino      Adultos         1  
2                 CONTUNDENTES   Femenino      Adultos         3  
3                 CONTUNDENTES  Masculino      Adultos         1  
4                     VEHICULO   Femenino      Adultos         1  

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2786613 entries, 0 to 2786612
Data columns (total 8 columns):
 #   Column        Dtype         
---  ------        -----         
 0   fecha_hecho   datetime64[ns]
 1   depar

##### **1.1. Construcción de la dimensión Tiempo**

La dimensión de tiempo se construye a partir de las fechas únicas presentes en el conjunto de datos. Para cada fecha se generan los atributos temporales que facilitarán el análisis de la información desde diferentes perspectivas cronológicas, incluyendo día, mes, año, trimestre y la identificación de fines de semana.

In [4]:
# Construir la dimensión tiempo
dim_tiempo = (
    df[["fecha_hecho"]]
    .drop_duplicates()
    .sort_values("fecha_hecho")
    .reset_index(drop=True)
)

# Diccionarios para nombres
dias = {
    0: "Lunes",
    1: "Martes",
    2: "Miércoles",
    3: "Jueves",
    4: "Viernes",
    5: "Sábado",
    6: "Domingo"
}

meses = {
    1: "Enero",
    2: "Febrero",
    3: "Marzo",
    4: "Abril",
    5: "Mayo",
    6: "Junio",
    7: "Julio",
    8: "Agosto",
    9: "Septiembre",
    10: "Octubre",
    11: "Noviembre",
    12: "Diciembre"
}

# Atributos
dim_tiempo["dia"] = dim_tiempo["fecha_hecho"].dt.day
dim_tiempo["dia_semana"] = dim_tiempo["fecha_hecho"].dt.weekday + 1
dim_tiempo["nombre_dia"] = dim_tiempo["fecha_hecho"].dt.weekday.map(dias)
dim_tiempo["es_fin_semana"] = dim_tiempo["fecha_hecho"].dt.weekday >= 5

dim_tiempo["mes"] = dim_tiempo["fecha_hecho"].dt.month
dim_tiempo["nombre_mes"] = dim_tiempo["mes"].map(meses)

dim_tiempo["trimestre"] = dim_tiempo["fecha_hecho"].dt.quarter
dim_tiempo["anio"] = dim_tiempo["fecha_hecho"].dt.year

dim_tiempo.head()

,fecha_hecho,dia,dia_semana,nombre_dia,es_fin_semana,mes,nombre_mes,trimestre,anio
0,2014-01-01,1,3,Miércoles,False,1,Enero,1,2014
1,2014-01-02,2,4,Jueves,False,1,Enero,1,2014
2,2014-01-03,3,5,Viernes,False,1,Enero,1,2014
3,2014-01-04,4,6,Sábado,True,1,Enero,1,2014
4,2014-01-05,5,7,Domingo,True,1,Enero,1,2014


In [5]:
dim_tiempo.shape

(4018, 9)

##### **1.2. Construcción de la dimensión Ubicación**

La dimensión de ubicación se construye a partir de las combinaciones únicas de departamento y municipio presentes en los registros. Esta dimensión permite analizar la ocurrencia de los delitos desde una perspectiva geográfica, facilitando consultas tanto a nivel departamental como municipal.

In [6]:
# Construir la dimensión ubicación

dim_ubicacion = (
    df[["departamento", "municipio"]]
    .drop_duplicates()
    .sort_values(["departamento", "municipio"])
    .reset_index(drop=True)
)

print(f"Registros: {len(dim_ubicacion):,}")

dim_ubicacion.head()

Registros: 1,107


,departamento,municipio
0,Amazonas,Leticia (Ct)
1,Amazonas,Puerto Nariño
2,Antioquia,Abejorral
3,Antioquia,Abriaquí
4,Antioquia,Alejandría


##### **1.3. Construcción de la dimensión Delito**

La dimensión de delito se construye a partir de los tipos de delito presentes en el conjunto de datos. Su propósito es clasificar los registros de la tabla de hechos según la naturaleza del delito, permitiendo realizar análisis comparativos entre las diferentes categorías consideradas en la investigación.

In [7]:
# Construir la dimensión delito

dim_delito = (
    df[["delito"]]
    .drop_duplicates()
    .sort_values("delito")
    .reset_index(drop=True)
)

print(f"Registros: {len(dim_delito)}")

dim_delito

Registros: 7


,delito
0,Amenazas
1,Delitos sexuales
2,Homicidio
3,Hurto a residencias y entidades comerciales
4,Hurto de motocicletas y automotores
5,Lesiones personales
6,Violencia intrafamiliar


##### **1.4. Construcción de la dimensión Arma**

La dimensión de arma se construye a partir de los valores únicos del atributo armas o medios. Esta dimensión permite analizar la distribución de los delitos según el arma o medio utilizado durante la ocurrencia del hecho.

In [8]:
# Construir la dimensión arma

dim_arma = (
    df[["armas_medios"]]
    .drop_duplicates()
    .sort_values("armas_medios")
    .reset_index(drop=True)
)

print(f"Registros: {len(dim_arma)}")

dim_arma.head(10)

Registros: 49


,armas_medios
0,ACIDO
1,AGUA CALIENTE
2,ALIMENTOS VENCIDOS
3,ALMOHADA
4,ALUCINOGENOS
5,ARMA BLANCA / CORTOPUNZANTE
6,ARMA DE FUEGO
7,ARMA TRAUMATICA
8,ARTEFACTO EXPLOSIVO/CARGA DINAMITA
9,ARTEFACTO INCENDIARIO


##### **1.5. Construcción de la dimensión Víctima**

La dimensión de víctima se construye a partir de las combinaciones únicas de género y grupo etario presentes en los registros. Esta dimensión permite caracterizar a las víctimas de los hechos delictivos y realizar análisis según sus características demográficas.

In [9]:
# Construir la dimensión víctima

dim_victima = (
    df[["genero", "grupo_etario"]]
    .drop_duplicates()
    .sort_values(["genero", "grupo_etario"])
    .reset_index(drop=True)
)

print(f"Registros: {len(dim_victima)}")

dim_victima

Registros: 10


,genero,grupo_etario
0,Femenino,Adolescentes
1,Femenino,Adultos
2,Femenino,Menores
3,Masculino,Adolescentes
4,Masculino,Adultos
5,Masculino,Menores
6,No reportado,Adolescentes
7,No reportado,Adultos
8,No reportado,Menores
9,No reportado,No reportado


##### **1.6. Validacion y carga**

In [10]:
print("Dim_Tiempo:", len(dim_tiempo))
print("Dim_Ubicacion:", len(dim_ubicacion))
print("Dim_Delito:", len(dim_delito))
print("Dim_Arma:", len(dim_arma))
print("Dim_Victima:", len(dim_victima))

Dim_Tiempo: 4018
Dim_Ubicacion: 1107
Dim_Delito: 7
Dim_Arma: 49
Dim_Victima: 10


In [11]:
print(dim_tiempo.duplicated().sum())
print(dim_ubicacion.duplicated().sum())
print(dim_delito.duplicated().sum())
print(dim_arma.duplicated().sum())
print(dim_victima.duplicated().sum())

0
0
0
0
0


Borrar contenido de tablas

In [12]:
from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE fact_delitos RESTART IDENTITY CASCADE;"))
    conn.execute(text("TRUNCATE TABLE dim_tiempo RESTART IDENTITY CASCADE;"))
    conn.execute(text("TRUNCATE TABLE dim_ubicacion RESTART IDENTITY CASCADE;"))
    conn.execute(text("TRUNCATE TABLE dim_delito RESTART IDENTITY CASCADE;"))
    conn.execute(text("TRUNCATE TABLE dim_arma RESTART IDENTITY CASCADE;"))
    conn.execute(text("TRUNCATE TABLE dim_victima RESTART IDENTITY CASCADE;"))

Homologaciones necesarias

In [13]:
# Dimensión Tiempo
dim_tiempo = dim_tiempo.rename(
    columns={
        "fecha_hecho": "fecha"
    }
)

# Dimensión Delito
dim_delito = dim_delito.rename(
    columns={
        "delito": "tipo_delito"
    }
)

# Dimensión Arma
dim_arma = dim_arma.rename(
    columns={
        "armas_medios": "nombre_arma"
    }
)

Carga de dim_tiempo

In [14]:
dim_tiempo.to_sql(
    "dim_tiempo",
    engine,
    if_exists="append",
    index=False
)

18

Carga de dim_ubicacion

In [15]:
dim_ubicacion.to_sql(
    "dim_ubicacion",
    engine,
    if_exists="append",
    index=False
)

107

Carga de dim_delito

In [16]:
dim_delito.to_sql(
    "dim_delito",
    engine,
    if_exists="append",
    index=False
)

7

Carga de dim_arma

In [17]:
dim_arma.to_sql(
    "dim_arma",
    engine,
    if_exists="append",
    index=False
)

49

Carga de dim_victima

In [18]:
dim_victima.to_sql(
    "dim_victima",
    engine,
    if_exists="append",
    index=False
)

10

### **2. Carga de tabla de hechos Delitos**

#### **2.1. Recuperación de las dimensiones**

Una vez cargadas las dimensiones en PostgreSQL, se consultan nuevamente para recuperar las claves sustitutas generadas automáticamente. Estas claves serán utilizadas para establecer las relaciones entre las dimensiones y la tabla de hechos durante la construcción del modelo estrella.

In [19]:
# Leer las dimensiones desde PostgreSQL

dim_tiempo_dw = pd.read_sql(
    "SELECT * FROM dim_tiempo",
    engine,
    parse_dates=["fecha"]
)

dim_ubicacion_dw = pd.read_sql(
    "SELECT * FROM dim_ubicacion",
    engine
)

dim_delito_dw = pd.read_sql(
    "SELECT * FROM dim_delito",
    engine
)

dim_arma_dw = pd.read_sql(
    "SELECT * FROM dim_arma",
    engine
)

dim_victima_dw = pd.read_sql(
    "SELECT * FROM dim_victima",
    engine
)

In [20]:
print(dim_tiempo_dw.head())

print("\n", dim_ubicacion_dw.head())

print("\n", dim_delito_dw.head())

print("\n", dim_arma_dw.head())

print("\n", dim_victima_dw.head())

   id_tiempo      fecha  dia  mes nombre_mes  trimestre  anio  dia_semana  \
0          1 2014-01-01    1    1      Enero          1  2014           3   
1          2 2014-01-02    2    1      Enero          1  2014           4   
2          3 2014-01-03    3    1      Enero          1  2014           5   
3          4 2014-01-04    4    1      Enero          1  2014           6   
4          5 2014-01-05    5    1      Enero          1  2014           7   

  nombre_dia  es_fin_semana  
0  Miércoles          False  
1     Jueves          False  
2    Viernes          False  
3     Sábado           True  
4    Domingo           True  

    id_ubicacion departamento      municipio
0             1     Amazonas   Leticia (Ct)
1             2     Amazonas  Puerto Nariño
2             3    Antioquia      Abejorral
3             4    Antioquia       Abriaquí
4             5    Antioquia     Alejandría

    id_delito                                  tipo_delito
0          1                   

#### **2.2. Relacionar dataframe principal con dimensiones**

Merge con dim tiempo

In [21]:
fact_delitos = df.merge(
    dim_tiempo_dw[["id_tiempo", "fecha"]],
    left_on="fecha_hecho",
    right_on="fecha",
    how="left"
)

print(f"Registros: {len(fact_delitos):,}")

Registros: 2,786,613


In [22]:
print("id_tiempo nulos:", fact_delitos["id_tiempo"].isna().sum())

id_tiempo nulos: 0


Merge con dim ubicacion

In [23]:
fact_delitos = fact_delitos.merge(
    dim_ubicacion_dw,
    on=["departamento", "municipio"],
    how="left"
)

In [24]:
print("id_ubicacion nulos:", fact_delitos["id_ubicacion"].isna().sum())

id_ubicacion nulos: 0


Merge con dim tipo delito

In [25]:
fact_delitos = fact_delitos.merge(
    dim_delito_dw,
    left_on="delito",
    right_on="tipo_delito",
    how="left"
)

In [26]:
print("id_delito nulos:", fact_delitos["id_delito"].isna().sum())

id_delito nulos: 0


Merge con dim arma

In [27]:
fact_delitos = fact_delitos.merge(
    dim_arma_dw,
    left_on="armas_medios",
    right_on="nombre_arma",
    how="left"
)

In [28]:
print("id_arma nulos:", fact_delitos["id_arma"].isna().sum())

id_arma nulos: 0


Merge con dim victima

In [29]:
fact_delitos = fact_delitos.merge(
    dim_victima_dw,
    on=["genero", "grupo_etario"],
    how="left"
)

In [30]:
print("id_victima nulos:", fact_delitos["id_victima"].isna().sum())

id_victima nulos: 0


#### **2.3. Validación y carga**

In [31]:
fact_delitos = fact_delitos[
    [
        "id_delito",
        "id_tiempo",
        "id_ubicacion",
        "id_victima",
        "id_arma",
        "cantidad"
    ]
].copy()

fact_delitos = fact_delitos.rename(
    columns={
        "cantidad": "numero"
    }
)

fact_delitos.head()

,id_delito,id_tiempo,id_ubicacion,id_victima,id_arma,numero
0,1,1,6,2,46,1
1,6,1,6,5,6,1
2,6,1,8,2,18,3
3,6,1,10,5,18,1
4,6,1,10,2,48,1


In [32]:
print(f"Registros de la tabla de hechos: {len(fact_delitos):,}")

print(f"Suma total de delitos: {fact_delitos['numero'].sum():,}")

Registros de la tabla de hechos: 2,786,613
Suma total de delitos: 4,805,136


In [33]:
print(fact_delitos.isna().sum())

id_delito       0
id_tiempo       0
id_ubicacion    0
id_victima      0
id_arma         0
numero          0
dtype: int64


In [34]:
fact_delitos = fact_delitos.rename(
    columns={
        "numero": "cantidad_delitos"
    }
)

fact_delitos.to_sql(
    "fact_delitos",
    engine,
    if_exists="append",
    index=False
)

print(f"Fact_Delitos cargada correctamente ({len(fact_delitos):,} registros).")

Fact_Delitos cargada correctamente (2,786,613 registros).
